# In-context learning, from zero

A learning notebook: build up in-context learning piece by piece, watching each idea land before adding the next.

In [2]:
import torch
import numpy as np

torch.manual_seed(42)
print("Torch device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Torch device: cuda


## Dataset — matching the paper, not TinyStories

Olsson et al. 2022 trained their small models on an earlier version of the corpus described in
Askell et al. 2021 (arXiv:2112.00861): **filtered Common Crawl + internet books + ~10% Python
code**, plus several smaller distributions. Askell et al.'s exact corpus was never released.

We use `monology/pile-uncopyrighted` instead: The Pile with the five subsets under copyright
takedown removed (Books3, BookCorpus2, OpenSubtitles, YTSubtitles, OWT2). What remains — Pile-CC
(filtered Common Crawl), Gutenberg (books), GitHub (code, ~7-8% of tokens in the original Pile mix
— close to the paper's ~10%), plus the same long tail of smaller distributions Askell et al.
describe — is the closest reliably-downloadable public match. It's also what Pythia, this
project's other reference model family, was trained on.

**Why this matters for what we're building:** the earlier `stage2c_induction_tinystories.ipynb`
notebook, trained on TinyStories, never reproduced the paper's sharp induction-score phase
transition — the rise was gradual instead. TinyStories has no code and only short documents;
code's near-verbatim repetition may be exactly what sharpens that signal in the paper. Training on
a Pile-like mix lets us actually test that, instead of assuming it.

In [3]:
DATASET_NAME = "monology/pile-uncopyrighted"  # near-identical to Olsson et al.'s corpus
CHAR_BUDGET = 200_000_000  # same order of magnitude as the stage2c TinyStories run
CACHE_DIR = "checkpoints/icl_pile"
CORPUS_CACHE = f"{CACHE_DIR}/corpus.txt"

print(f"[config] dataset={DATASET_NAME!r} char_budget={CHAR_BUDGET:,} cache={CORPUS_CACHE!r}")

[config] dataset='monology/pile-uncopyrighted' char_budget=200,000,000 cache='checkpoints/icl_pile/corpus.txt'


## Smoke test — does this actually stream, and is it what we expect?

Before pulling `CHAR_BUDGET` characters, pull just 5 rows and look at them. Each row is
`{"text": ..., "meta": {"pile_set_name": ...}}` — `pile_set_name` tells us which of the Pile's
original components a row came from, so we can sanity-check the mix (Common Crawl, code, books,
...) before committing to a full pull.

In [4]:
from datasets import load_dataset

_stream = load_dataset(DATASET_NAME, split="train", streaming=True)
_sample = [row for _, row in zip(range(5), _stream)]

for i, row in enumerate(_sample):
    source = row["meta"]["pile_set_name"]
    preview = row["text"][:100].replace("\n", " ")
    print(f"[{i}] source={source!r}")
    print(f"    {preview!r}")

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

[0] source='Pile-CC'
    'It is done, and submitted. You can play “Survival of the Tastiest” on Android, and on the web. Playi'
[1] source='Github'
    '<?xml version="1.0" encoding="UTF-8"?>\r <segment>\r     <name>PD1</name>\r     <description>Patient Ad'
[2] source='Pile-CC'
    'Topic: reinvent midnight madness  Amazon announced a new service at the AWS re:Invent Midnight Madne'
[3] source='Pile-CC'
    'About Grand Slam Fishing Charters  As a family owned business we know how important it is that your '
[4] source='StackExchange'
    "Q:  Why was Mundungus banned from the Hog's Head?  In Order of the Phoenix while the trio were in th"
